# YOLO26s 차량 차종 분류 학습/예측 Colab

이 노트북은 `data/vehicle_cls_crops_sampled` 형태의 YOLO classification 데이터셋을 Colab GPU에서 학습하고, 차량 이미지에 대해 차종 예측을 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 0. GPU와 패키지 준비

In [ ]:
!nvidia-smi
!pip install -q -U ultralytics opencv-python pillow pyyaml tqdm

Fri Jul  3 02:52:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             27W /   70W |     137MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Google Drive 마운트와 경로 설정

`car-cls` 프로젝트 폴더를 Drive에 올린 뒤 `PROJECT_DIR`만 맞추면 됩니다. 이미 정제된 데이터셋도 같이 올렸다면 기본값 그대로 사용하세요.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

PROJECT_ZIP = '/content/drive/MyDrive/car-cls.zip'

# 프로젝트를 풀 위치
PROJECT_DIR = '/content/car-cls'

# 데이터셋 위치
DATASET_LOCAL = f'{PROJECT_DIR}/data/vehicle_cls_crops_sampled'

RUNS_DRIVE = '/content/drive/MyDrive/runs/vehicle_cls'
TEST_CROPS = '/content/drive/MyDrive/test_vehicle_crops'
TEST_FULL_IMAGES = '/content/drive/MyDrive/test_vehicle_images'

assert Path(PROJECT_ZIP).exists(), f'Zip file does not exist: {PROJECT_ZIP}'

# 이전 프로젝트만 삭제
!rm -rf "$PROJECT_DIR"

# 프로젝트 폴더 생성
!mkdir -p "$PROJECT_DIR"

# 압축 해제
!unzip -o -q "$PROJECT_ZIP" -d "$PROJECT_DIR"

Mounted at /content/drive


## 2. 데이터셋을 Colab 로컬 디스크로 복사 - 생략

Drive에서 직접 학습하면 느릴 수 있으므로, 학습할 때는 `/content`로 복사해서 사용합니다. 결과 weights는 Drive에 저장됩니다.

In [ ]:
from pathlib import Path

DATASET_LOCAL = Path(DATASET_LOCAL)

for split in ['train', 'val', 'test']:
    split_dir = DATASET_LOCAL / split
    image_count = len([
        p for p in split_dir.rglob('*')
        if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']
    ])
    class_count = len([p for p in split_dir.iterdir() if p.is_dir()])
    print(split, 'classes=', class_count, 'images=', image_count)

train classes= 188 images= 51330
val classes= 188 images= 9620
test classes= 188 images= 3207


## 3. YOLO26s classification 학습

Colab 무료 GPU 기준으로 처음에는 `epochs=30` 정도로 빠르게 확인하고, 결과가 괜찮으면 `50~100`으로 늘리면 됩니다.

In [ ]:
!find /content/drive/MyDrive/runs/vehicle_cls -path "*/weights/best.pt"

/content/drive/MyDrive/runs/vehicle_cls/yolo26s_cls_full/weights/best.pt
/content/drive/MyDrive/runs/vehicle_cls/yolo26s_cls_full-11/weights/best.pt
/content/drive/MyDrive/runs/vehicle_cls/yolo26s_cls_full-12/weights/best.pt
/content/drive/MyDrive/runs/vehicle_cls/yolo26s_cls_full-15/weights/best.pt


In [ ]:
import glob
from pathlib import Path
import os # os 모듈 추가

# --- 학습 재개(Resume) 설정 ---
# SHOULD_RESUME을 True로 설정하면 중단된 학습을 자동으로 이어갑니다.
# False로 설정하면 yolo26s-cls.pt 모델로 새롭게 학습을 시작합니다.
SHOULD_RESUME = True # @param {type:"boolean"}

# 특정 last.pt 파일에서 이어서 학습하고 싶다면 여기에 경로를 지정하세요.
# 예시: '/content/drive/MyDrive/runs/vehicle_cls/yolo26s_cls_full-8/weights/last.pt'
EXPLICIT_RESUME_PATH = '' # @param {type:"string"}

training_model = 'yolo26s-cls.pt'
resume_argument = '' # 이 변수는 이제 사용하지 않음
run_name = 'yolo26s_cls_full' # 현재 학습의 기본 이름

if SHOULD_RESUME:
    resume_path = EXPLICIT_RESUME_PATH
    if not resume_path:
        # 'run_name'과 일치하는 가장 최근의 학습 폴더에서 last.pt를 자동으로 찾습니다.
        # 예: yolo26s_cls_full, yolo26s_cls_full-1, yolo26s_cls_full-2 등
        base_run_dir_pattern = Path(RUNS_DRIVE) / run_name
        # glob.glob은 문자열 경로를 반환하므로 os.path.getmtime을 사용합니다.
        all_run_dirs = sorted(glob.glob(f'{base_run_dir_pattern}*'), key=os.path.getmtime, reverse=True)

        if all_run_dirs:
            latest_run_dir = Path(all_run_dirs[0])
            potential_last_pt = latest_run_dir / 'weights' / 'last.pt'
            if potential_last_pt.exists():
                resume_path = str(potential_last_pt)
                print(f"가장 최근 학습의 last.pt를 자동으로 찾았습니다: {resume_path}")
            else:
                print(f"가장 최근 학습 폴더({latest_run_dir})에서 last.pt를 찾을 수 없습니다. 새롭게 학습을 시작합니다.")
        else:
            print(f"'{run_name}' 이름으로 이전 학습 기록을 찾을 수 없습니다. 새롭게 학습을 시작합니다.")

    if resume_path and Path(resume_path).exists():
        training_model = resume_path # training_model에 last.pt 경로를 직접 할당
        print(f"'{training_model}'에서 학습을 재개합니다.")
    else:
        print("학습을 재개하지 않습니다. 새 모델로 시작합니다.")
else:
    print("SHOULD_RESUME이 False로 설정되어 있습니다. 새 모델로 학습을 시작합니다.")

# --- 학습 명령어 ---
# batch 사이즈 (-1은 자동 설정, 32, 64와 같이 명시적인 숫자 권장).
# OOM(메모리 부족)이나 잦은 중단이 발생할 경우 batch 사이즈를 줄여보세요.
# Ultralytics는 --model에 last.pt 경로를 지정하면 자동으로 학습을 재개합니다.
!python /content/car-cls/src/train.py --task classify --model "$training_model" --data "$DATASET_LOCAL" --epochs 50 --imgsz 224 --batch 32 --project "$RUNS_DRIVE" --name "$run_name" --device 0 --workers 2 --patience 5


가장 최근 학습의 last.pt를 자동으로 찾았습니다: /content/drive/MyDrive/runs/vehicle_cls/yolo26s_cls_full-15/weights/last.pt
'/content/drive/MyDrive/runs/vehicle_cls/yolo26s_cls_full-15/weights/last.pt'에서 학습을 재개합니다.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/car-cls/data/vehicle_cls_crops_sampled, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, di

## 4. 검증 지표 확인

학습이 끝나면 best weights로 validation set 성능을 다시 확인합니다.

In [ ]:
!pip install -q ultralytics opencv-python pyyaml tqdm "pillow==11.3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 97.3 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

CLS_WEIGHTS = f'{RUNS_DRIVE}/yolo26s_cls_full-16/weights/best.pt'
model = YOLO(CLS_WEIGHTS, task='classify')
metrics = model.val(data=str(DATASET_LOCAL), imgsz=224, device=0)
metrics

Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s-cls summary (fused): 47 layers, 5,674,956 parameters, 0 gradients, 12.2 GFLOPs
train: /content/car-cls/data/vehicle_cls_crops_sampled/train... found 51330 images in 188 classes ✅ 
val: /content/car-cls/data/vehicle_cls_crops_sampled/val... found 9620 images in 188 classes ✅ 
test: /content/car-cls/data/vehicle_cls_crops_sampled/test... found 3207 images in 188 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 782.8±361.6 MB/s, size: 25.2 KB)
val: Scanning /content/car-cls/data/vehicle_cls_crops_sampled/val... 9620 images, 0 corrupt: 100% ━━━━━━━━━━━━ 9620/9620 5.7Kit/s 1.7s
val: New cache created: /content/car-cls/data/vehicle_cls_crops_sampled/val.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 602/602 16.1it/s 37.3s
                   all      0.914      0.986
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /con

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c519292ff20>
curves: []
curves_results: []
fitness: 0.9498960375785828
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9142411351203918, 'metrics/accuracy_top5': 0.9855509400367737, 'fitness': 0.9498960375785828}
save_dir: PosixPath('/content/runs/classify/val-2')
speed: {'preprocess': 0.0742932243233535, 'inference': 1.261154825260557, 'loss': 0.0018217153847917172, 'postprocess': 0.002262107900482865}
top1: 0.9142411351203918
top5: 0.9855509400367737

## 5. Crop된 차량 이미지 차종 예측

`TEST_CROPS`에 차량만 crop된 이미지들을 넣으면 top-5 예측을 CSV로 저장합니다.

In [48]:
!python "$PROJECT_DIR/src/predict_cls.py" \
  --weights "$CLS_WEIGHTS" \
  --source "$TEST_CROPS" \
  --out /content/drive/MyDrive/runs/vehicle_cls_predict \
  --imgsz 224 \
  --topk 5 \
  --device 0

Images predicted: 10
Predictions CSV: /content/drive/MyDrive/runs/vehicle_cls_predict/predictions.csv


In [ ]:
import pandas as pd

pred_csv = '/content/drive/MyDrive/runs/vehicle_cls_predict/predictions.csv'
pd.read_csv(pred_csv).head(20)

,image,top1_class,top1_conf,top2_class,top2_conf,top3_class,top3_conf,top4_class,top4_conf,top5_class,top5_conf
0,/content/drive/MyDrive/test_vehicle_crops/3_se...,starex_grand,0.232285,porter_2_hr,0.190047,grandeur_tg,0.180361,bongo_bongo_III,0.090896,starex_grand_the_new,0.027161
1,/content/drive/MyDrive/test_vehicle_crops/C-22...,5_series_f10,0.999874,3_series_f30,0.000081,5_series_g30,0.000035,4_series_f32,0.000008,1_series_f20,0.000001
2,/content/drive/MyDrive/test_vehicle_crops/K5-2...,k5_2th,0.994632,sorento_4th,0.001282,camry_xv70,0.000556,k3_all_new,0.000449,tivoli_very_new,0.000435
3,/content/drive/MyDrive/test_vehicle_crops/SONA...,sonata_yf,0.883254,grandeur_hg,0.065596,sonata_lf,0.032129,k7_k7,0.005742,grandeur_ig,0.001553
4,/content/drive/MyDrive/test_vehicle_crops/SORE...,sorento_4th,0.980701,palisade_palisade,0.003627,seltos_seltos,0.002888,carnival_4th,0.002329,discovery_sports,0.001061
5,/content/drive/MyDrive/test_vehicle_crops/prid...,pride_all_new,0.792650,sportage_4th,0.110546,k3_k3,0.014851,k5_the_new,0.012130,morning_urban,0.011973
6,/content/drive/MyDrive/test_vehicle_crops/seri...,5_series_f10,0.812317,alpheon_alpheon,0.075608,x_series_x5_g05,0.059973,opirus_premium,0.017337,5_series_g30,0.009989
7,/content/drive/MyDrive/test_vehicle_crops/seri...,x_series_x5_g05,0.893168,1_series_f20,0.102606,5_series_f10,0.002895,cooper_countryman,0.000668,glc_class_x253,0.000188
8,/content/drive/MyDrive/test_vehicle_crops/seri...,3_series_f30,0.597228,4_series_f32,0.223622,5_series_f10,0.106319,grandeur_ig,0.029438,sm5_sm5,0.008833
9,/content/drive/MyDrive/test_vehicle_crops/star...,starex_grand,0.995843,starex_starex,0.002098,sm7_sm7,0.000566,carnival_r,0.000137,i30_new,0.000118


## 6. 일반 차량 사진에서 탐지 후 차종 예측

`TEST_FULL_IMAGES`에 도로/주차장 사진처럼 차량이 포함된 원본 이미지를 넣으면, COCO pretrained `yolo26s.pt`로 car/truck을 먼저 탐지하고 crop을 분류합니다.

In [ ]:
!python "$PROJECT_DIR/src/infer_two_stage.py" \
  --det-weights yolo26s.pt \
  --cls-weights "$CLS_WEIGHTS" \
  --source "$TEST_FULL_IMAGES" \
  --out /content/drive/MyDrive/runs/two_stage_vehicle_cls \
  --det-conf 0.25 \
  --cls-imgsz 224

[DEBUG] Processing 188 images from /content/drive/MyDrive/test_vehicle_images
Predictions written: /content/drive/MyDrive/runs/two_stage_vehicle_cls/predictions.csv
Annotated images: /content/drive/MyDrive/runs/two_stage_vehicle_cls/annotated
Crops: /content/drive/MyDrive/runs/two_stage_vehicle_cls/crops


In [ ]:
two_stage_csv = '/content/drive/MyDrive/runs/two_stage_vehicle_cls/predictions.csv'
pd.read_csv(two_stage_csv).head(299)

: 

먼저 `TEST_FULL_IMAGES` 경로에 실제로 몇 개의 이미지 파일이 존재하는지 확인해 보겠습니다.

In [ ]:
from pathlib import Path

image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_count = 0

# Recursively count image files in TEST_FULL_IMAGES and its subdirectories
for item in Path(TEST_FULL_IMAGES).rglob('*'):
    if item.is_file() and item.suffix.lower() in image_extensions:
        image_count += 1

print(f"'{TEST_FULL_IMAGES}' 경로에 있는 이미지 파일의 총 개수: {image_count}")

# Additionally, let's list the first few image files found, to ensure the path is correct
print("\n처음 몇 개의 이미지 파일 (확인용):")
count_listed = 0
for item in Path(TEST_FULL_IMAGES).rglob('*'):
    if item.is_file() and item.suffix.lower() in image_extensions:
        print(item)
        count_listed += 1
        if count_listed >= 5: # List up to 5 files
            break
if count_listed == 0:
    print("이미지 파일을 찾을 수 없습니다.")

'/content/drive/MyDrive/test_vehicle_images' 경로에 있는 이미지 파일의 총 개수: 188

처음 몇 개의 이미지 파일 (확인용):
/content/drive/MyDrive/test_vehicle_images/glc_class_x253.jpg
/content/drive/MyDrive/test_vehicle_images/carnival_all_new.jpg
/content/drive/MyDrive/test_vehicle_images/ev6_ev6.jpg
/content/drive/MyDrive/test_vehicle_images/grandeur_xg.jpg
/content/drive/MyDrive/test_vehicle_images/i30_new.jpg


예상대로 `TEST_FULL_IMAGES` 경로에는 188개의 이미지가 있습니다. 하지만 `infer_two_stage.py` 스크립트는 9개의 이미지만 처리했습니다. `infer_two_stage.py` 스크립트의 내용을 확인하여 왜 모든 이미지를 처리하지 않는지 알아보겠습니다.

In [ ]:
with open(f'{PROJECT_DIR}/src/infer_two_stage.py', 'r') as f:
    script_content = f.read()

# Add a print statement to verify the number of images being processed
modified_script_content = script_content.replace(
    "    for image_path in iter_images(args.source):",
    "    image_paths = iter_images(args.source)\n    print(f\"[DEBUG] Processing {len(image_paths)} images from {args.source}\")\n    for image_path in image_paths:"
)

with open(f'{PROJECT_DIR}/src/infer_two_stage.py', 'w') as f:
    f.write(modified_script_content)

print("infer_two_stage.py 스크립트에 디버그용 출력문이 추가되었습니다. 이제 스크립트를 다시 실행해 주세요.")

infer_two_stage.py 스크립트에 디버그용 출력문이 추가되었습니다. 이제 스크립트를 다시 실행해 주세요.
